# Sinkhorn Aligner: Pallas Backward Pass Optimization Benchmark
This notebook compares the original Multi-Transport-Plan backward kernels with the new mathematically optimized ones that factor out `safe_exp`.

In [15]:
!pip install --upgrade "jax[tpu]" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html


Looking in links: https://storage.googleapis.com/jax-releases/libtpu_releases.html


In [16]:
import jax
import jax.numpy as jnp
from jax.experimental import pallas as pl
import time

try:
    from jax.experimental.pallas import tpu as pltpu
except ImportError:
    pltpu = None

def resolve_compiler_params(dim_semantics):
    if pltpu is None:
        return None
    if hasattr(pltpu, "CompilerParams"):
        return pltpu.CompilerParams(dimension_semantics=dim_semantics)
    elif hasattr(pltpu, "TPUCompilerParams"):
        return pltpu.TPUCompilerParams(dimension_semantics=dim_semantics)
    return None

def _poly_exp2_frac(r):
    p5 = 1.5252715e-4
    p4 = 1.3215486e-3
    p3 = 9.6178085e-3
    p2 = 5.5504109e-2
    p1 = 2.4022651e-1
    p0 = 6.9314718e-1
    return 1.0 + r * (p0 + r * (p1 + r * (p2 + r * (p3 + r * (p4 + r * p5)))))

def safe_exp(x):
    LOG2E = jnp.float32(1.4426950408889634)
    t = x * LOG2E
    k = jnp.floor(t)
    r = t - k
    k = jnp.clip(k, -126.0, 127.0)
    pow2_int = jnp.exp2(k)
    pow2_frac = _poly_exp2_frac(r)
    return pow2_int * pow2_frac


## 1. Original Backward Pass Kernels

In [17]:
def dq_kernel_multi_orig(
    Q_ref, K_ref, V_ref, M_data_ref, N_data_ref, dO_ref, W_ref, dQ_ref
):
    q_block = Q_ref[...]
    do_block = dO_ref[...]
    m_data = M_data_ref[...]
    W = W_ref[0]

    u0_block = m_data[:, 0]
    u1_block = m_data[:, 1]
    u2_block = m_data[:, 2]
    del_u1_block = m_data[:, 3]
    del_u2_block = m_data[:, 4]

    D = q_block.shape[-1]
    scale = 1.0 / jnp.sqrt(D)

    N = K_ref.shape[1]
    BLOCK_N = 128
    UNROLL = 4
    iters = N // BLOCK_N

    m = pl.program_id(0)
    start_block = jnp.maximum(0, m - W)
    end_block = jnp.minimum(iters, m + W + 1)
    start_iter = start_block // UNROLL
    end_iter = (end_block + UNROLL - 1) // UNROLL

    init_state = (jnp.zeros_like(q_block),)

    def body_fn(i, state):
        (dq_acc,) = state
        for u_i in range(UNROLL):
            k_idx = i * UNROLL + u_i
            off = k_idx * BLOCK_N
            mask = (k_idx >= start_block) & (k_idx < end_block)

            k_block = K_ref[pl.ds(off, BLOCK_N), :]
            v_mat_block = V_ref[pl.ds(off, BLOCK_N), :]
            n_data = N_data_ref[pl.ds(off, BLOCK_N), :]

            v0_block = n_data[:, 0]
            v1_block = n_data[:, 1]
            v2_block = n_data[:, 2]
            g_v_block = n_data[:, 3]
            del_v1_block = n_data[:, 4]

            s_block = jnp.einsum("md,nd->mn", q_block, k_block) * scale

            P22 = safe_exp(s_block + u2_block[:, None] + v2_block[None, :])
            P21 = safe_exp(s_block + u2_block[:, None] + v1_block[None, :])
            P11 = safe_exp(s_block + u1_block[:, None] + v1_block[None, :])
            P10 = safe_exp(s_block + u1_block[:, None] + v0_block[None, :])

            z_block = jnp.einsum("me,ne->mn", do_block, v_mat_block)

            ds_block = (
                P22 * z_block
                - P22 * g_v_block[None, :]
                - P21 * del_u2_block[:, None]
                - P11 * del_v1_block[None, :]
                - P10 * del_u1_block[:, None]
            )

            ds_block = jnp.where(mask, ds_block, 0.0)
            dq_acc = dq_acc + jnp.einsum("mn,nd->md", ds_block, k_block)
        return (dq_acc,)

    res = jax.lax.fori_loop(start_iter, end_iter, body_fn, init_state)
    dq_acc = res[0]
    dQ_ref[...] = dq_acc * scale

def dk_dv_kernel_multi_orig(
    Q_ref, K_ref, V_ref, M_data_ref, N_data_ref, dO_ref, W_ref, dK_ref, dV_ref
):
    k_block = K_ref[...]
    v_mat_block = V_ref[...]
    n_data = N_data_ref[...]
    W = W_ref[0]

    v0_block_n = n_data[:, 0]
    v1_block_n = n_data[:, 1]
    v2_block_n = n_data[:, 2]
    g_v_block_n = n_data[:, 3]
    del_v1_block_n = n_data[:, 4]

    D = k_block.shape[-1]
    M = Q_ref.shape[1]
    BLOCK_M = 128
    UNROLL = 4
    iters = M // BLOCK_M
    scale = 1.0 / jnp.sqrt(D)

    n = pl.program_id(0)
    start_block = jnp.maximum(0, n - W)
    end_block = jnp.minimum(iters, n + W + 1)
    start_iter = start_block // UNROLL
    end_iter = (end_block + UNROLL - 1) // UNROLL

    init_state = (jnp.zeros_like(k_block), jnp.zeros_like(v_mat_block))

    def body_fn(i, state):
        dk_acc, dv_acc = state
        for u_i in range(UNROLL):
            m_idx = i * UNROLL + u_i
            off = m_idx * BLOCK_M
            mask = (m_idx >= start_block) & (m_idx < end_block)

            q_block = Q_ref[pl.ds(off, BLOCK_M), :]
            do_block = dO_ref[pl.ds(off, BLOCK_M), :]
            m_data = M_data_ref[pl.ds(off, BLOCK_M), :]

            u0_block = m_data[:, 0]
            u1_block = m_data[:, 1]
            u2_block = m_data[:, 2]
            del_u1_block = m_data[:, 3]
            del_u2_block = m_data[:, 4]

            s_block_T = jnp.einsum("nd,md->nm", k_block, q_block) * scale

            P22_T = safe_exp(s_block_T + v2_block_n[:, None] + u2_block[None, :])
            P21_T = safe_exp(s_block_T + v1_block_n[:, None] + u2_block[None, :])
            P11_T = safe_exp(s_block_T + v1_block_n[:, None] + u1_block[None, :])
            P10_T = safe_exp(s_block_T + v0_block_n[:, None] + u1_block[None, :])

            z_block_T = jnp.einsum("ne,me->nm", v_mat_block, do_block)

            ds_block_T = (
                P22_T * z_block_T
                - P22_T * g_v_block_n[:, None]
                - P21_T * del_u2_block[None, :]
                - P11_T * del_v1_block_n[:, None]
                - P10_T * del_u1_block[None, :]
            )

            ds_block_T = jnp.where(mask, ds_block_T, 0.0)
            P22_T = jnp.where(mask, P22_T, 0.0)

            dk_acc = dk_acc + jnp.einsum("nm,md->nd", ds_block_T, q_block)
            dv_acc = dv_acc + jnp.einsum("nm,me->ne", P22_T, do_block)
        return (dk_acc, dv_acc)

    res = jax.lax.fori_loop(start_iter, end_iter, body_fn, init_state)
    dk_acc, dv_acc = res
    dK_ref[...] = dk_acc * scale
    dV_ref[...] = dv_acc


## 2. Optimized Backward Pass Kernels

In [18]:
def dq_kernel_multi_opt(
    Q_ref, K_ref, V_ref, M_data_ref, N_data_ref, dO_ref, W_ref, dQ_ref
):
    q_block = Q_ref[...]
    do_block = dO_ref[...]
    m_data = M_data_ref[...]
    W = W_ref[0]

    u0_block = m_data[:, 0]
    u1_block = m_data[:, 1]
    u2_block = m_data[:, 2]
    del_u1_block = m_data[:, 3]
    del_u2_block = m_data[:, 4]

    D = q_block.shape[-1]
    scale = 1.0 / jnp.sqrt(D)

    N = K_ref.shape[1]
    BLOCK_N = 128
    UNROLL = 4
    iters = N // BLOCK_N

    m = pl.program_id(0)
    start_block = jnp.maximum(0, m - W)
    end_block = jnp.minimum(iters, m + W + 1)
    start_iter = start_block // UNROLL
    end_iter = (end_block + UNROLL - 1) // UNROLL

    init_state = (jnp.zeros_like(q_block),)

    def body_fn(i, state):
        (dq_acc,) = state
        for u_i in range(UNROLL):
            k_idx = i * UNROLL + u_i
            off = k_idx * BLOCK_N
            mask = (k_idx >= start_block) & (k_idx < end_block)

            k_block = K_ref[pl.ds(off, BLOCK_N), :]
            v_mat_block = V_ref[pl.ds(off, BLOCK_N), :]
            n_data = N_data_ref[pl.ds(off, BLOCK_N), :]

            v0_block = n_data[:, 0]
            v1_block = n_data[:, 1]
            v2_block = n_data[:, 2]
            g_v_block = n_data[:, 3]
            del_v1_block = n_data[:, 4]

            s_block = jnp.einsum("md,nd->mn", q_block, k_block) * scale

            # OPTIMIZATION: Compute only P11 with safe_exp, use normal exp for 1D vectors
            P11_opt = safe_exp(s_block + u1_block[:, None] + v1_block[None, :])
            exp_du21 = jnp.exp(u2_block - u1_block)
            exp_dv21 = jnp.exp(v2_block - v1_block)
            exp_dv01 = jnp.exp(v0_block - v1_block)
            
            term_P22 = exp_du21[:, None] * exp_dv21[None, :]
            term_P21 = exp_du21[:, None]
            term_P10 = exp_dv01[None, :]

            z_block = jnp.einsum("me,ne->mn", do_block, v_mat_block)

            ds_block = P11_opt * (
                term_P22 * z_block
                - term_P22 * g_v_block[None, :]
                - term_P21 * del_u2_block[:, None]
                - del_v1_block[None, :]
                - term_P10 * del_u1_block[:, None]
            )

            ds_block = jnp.where(mask, ds_block, 0.0)
            dq_acc = dq_acc + jnp.einsum("mn,nd->md", ds_block, k_block)
        return (dq_acc,)

    res = jax.lax.fori_loop(start_iter, end_iter, body_fn, init_state)
    dq_acc = res[0]
    dQ_ref[...] = dq_acc * scale

def dk_dv_kernel_multi_opt(
    Q_ref, K_ref, V_ref, M_data_ref, N_data_ref, dO_ref, W_ref, dK_ref, dV_ref
):
    k_block = K_ref[...]
    v_mat_block = V_ref[...]
    n_data = N_data_ref[...]
    W = W_ref[0]

    v0_block_n = n_data[:, 0]
    v1_block_n = n_data[:, 1]
    v2_block_n = n_data[:, 2]
    g_v_block_n = n_data[:, 3]
    del_v1_block_n = n_data[:, 4]

    D = k_block.shape[-1]
    M = Q_ref.shape[1]
    BLOCK_M = 128
    UNROLL = 4
    iters = M // BLOCK_M
    scale = 1.0 / jnp.sqrt(D)

    n = pl.program_id(0)
    start_block = jnp.maximum(0, n - W)
    end_block = jnp.minimum(iters, n + W + 1)
    start_iter = start_block // UNROLL
    end_iter = (end_block + UNROLL - 1) // UNROLL

    init_state = (jnp.zeros_like(k_block), jnp.zeros_like(v_mat_block))

    def body_fn(i, state):
        dk_acc, dv_acc = state
        for u_i in range(UNROLL):
            m_idx = i * UNROLL + u_i
            off = m_idx * BLOCK_M
            mask = (m_idx >= start_block) & (m_idx < end_block)

            q_block = Q_ref[pl.ds(off, BLOCK_M), :]
            do_block = dO_ref[pl.ds(off, BLOCK_M), :]
            m_data = M_data_ref[pl.ds(off, BLOCK_M), :]

            u0_block = m_data[:, 0]
            u1_block = m_data[:, 1]
            u2_block = m_data[:, 2]
            del_u1_block = m_data[:, 3]
            del_u2_block = m_data[:, 4]

            s_block_T = jnp.einsum("nd,md->nm", k_block, q_block) * scale

            # OPTIMIZATION: Factor out P11_T
            P11_T_opt = safe_exp(s_block_T + v1_block_n[:, None] + u1_block[None, :])
            exp_du21 = jnp.exp(u2_block - u1_block)
            exp_dv21 = jnp.exp(v2_block_n - v1_block_n)
            exp_dv01 = jnp.exp(v0_block_n - v1_block_n)

            term_P22_T = exp_dv21[:, None] * exp_du21[None, :]
            term_P21_T = exp_du21[None, :]
            term_P10_T = exp_dv01[:, None]

            z_block_T = jnp.einsum("ne,me->nm", v_mat_block, do_block)

            ds_block_T = P11_T_opt * (
                term_P22_T * z_block_T
                - term_P22_T * g_v_block_n[:, None]
                - term_P21_T * del_u2_block[None, :]
                - del_v1_block_n[:, None]
                - term_P10_T * del_u1_block[None, :]
            )
            P22_T = P11_T_opt * term_P22_T

            ds_block_T = jnp.where(mask, ds_block_T, 0.0)
            P22_T = jnp.where(mask, P22_T, 0.0)

            dk_acc = dk_acc + jnp.einsum("nm,md->nd", ds_block_T, q_block)
            dv_acc = dv_acc + jnp.einsum("nm,me->ne", P22_T, do_block)
        return (dk_acc, dv_acc)

    res = jax.lax.fori_loop(start_iter, end_iter, body_fn, init_state)
    dk_acc, dv_acc = res
    dK_ref[...] = dk_acc * scale
    dV_ref[...] = dv_acc


## 3. Dispatch Wrappers & Benchmarking Logic

In [19]:

def pallas_sinkhorn_bwd_fused_launcher(Q, K, V, dO, M_data, N_data, band_width, dq_kernel, dk_dv_kernel):
    # This wrapper sets up the grids and block specs without the extra Batch dimension,
    # and we will vmap it over the batch dimensions (B, H) externally.
    # Therefore, Q, K, V, dO, M_data, N_data are shapes (M, D), (N, D), (N, E), etc.
    M, D = Q.shape
    N, E = V.shape
    C = 8

    BLOCK = 128
    W_val = band_width // BLOCK if band_width is not None else M // BLOCK
    W_arr = jnp.array([W_val], dtype=jnp.int32)

    is_cpu = jax.devices()[0].platform == "cpu"
    kw = (
        {"interpret": True}
        if is_cpu
        else {}
    )

    grid_dq = (M // BLOCK,)
    dQ_f = pl.pallas_call(
        dq_kernel,
        out_shape=jax.ShapeDtypeStruct(Q.shape, Q.dtype),
        grid=grid_dq,
        in_specs=[
            pl.BlockSpec(index_map=lambda m: (m, 0), block_shape=(BLOCK, D)),
            pl.BlockSpec(index_map=lambda m: (0, 0), block_shape=(N, D)),
            pl.BlockSpec(index_map=lambda m: (0, 0), block_shape=(N, E)),
            pl.BlockSpec(index_map=lambda m: (m, 0), block_shape=(BLOCK, C)),
            pl.BlockSpec(index_map=lambda m: (0, 0), block_shape=(N, C)),
            pl.BlockSpec(index_map=lambda m: (m, 0), block_shape=(BLOCK, E)),
            pl.BlockSpec(index_map=lambda m: (0,), block_shape=(1,)),
        ],
        out_specs=pl.BlockSpec(index_map=lambda m: (m, 0), block_shape=(BLOCK, D)),
        **kw,
    )(Q, K, V, M_data, N_data, dO, W_arr)

    grid_dk_dv = (N // BLOCK,)
    out_shape_k = jax.ShapeDtypeStruct(K.shape, K.dtype)
    out_shape_v = jax.ShapeDtypeStruct(V.shape, V.dtype)

    dK_f, dV_f = pl.pallas_call(
        dk_dv_kernel,
        out_shape=(out_shape_k, out_shape_v),
        grid=grid_dk_dv,
        in_specs=[
            pl.BlockSpec(index_map=lambda n: (0, 0), block_shape=(M, D)),
            pl.BlockSpec(index_map=lambda n: (n, 0), block_shape=(BLOCK, D)),
            pl.BlockSpec(index_map=lambda n: (n, 0), block_shape=(BLOCK, E)),
            pl.BlockSpec(index_map=lambda n: (0, 0), block_shape=(M, C)),
            pl.BlockSpec(index_map=lambda n: (n, 0), block_shape=(BLOCK, C)),
            pl.BlockSpec(index_map=lambda n: (0, 0), block_shape=(M, E)),
            pl.BlockSpec(index_map=lambda n: (0,), block_shape=(1,)),
        ],
        out_specs=[
            pl.BlockSpec(index_map=lambda n: (n, 0), block_shape=(BLOCK, D)),
            pl.BlockSpec(index_map=lambda n: (n, 0), block_shape=(BLOCK, E)),
        ],
        **kw,
    )(Q, K, V, M_data, N_data, dO, W_arr)

    return dQ_f, dK_f, dV_f

def run_original(Q, K, V, dO, M_data, N_data, band_width=256):
    B, H, M, D = Q.shape
    C = M_data.shape[-1]
    _Q = Q.reshape(B*H, M, D)
    _K = K.reshape(B*H, K.shape[2], D)
    _V = V.reshape(B*H, V.shape[2], V.shape[3])
    _dO = dO.reshape(B*H, dO.shape[2], dO.shape[3])
    _M_data = M_data.reshape(B*H, M, C)
    _N_data = N_data.reshape(B*H, N_data.shape[2], C)
    
    # Use jax.vmap to handle the batch dimension across all arrays!
    fn = lambda q, k, v, do, m_data, n_data: pallas_sinkhorn_bwd_fused_launcher(
        q, k, v, do, m_data, n_data, band_width, dq_kernel_multi_orig, dk_dv_kernel_multi_orig
    )
    dQ, dK, dV = jax.lax.map(lambda inputs: fn(*inputs), (_Q, _K, _V, _dO, _M_data, _N_data))
    
    return dQ.reshape(Q.shape), dK.reshape(K.shape), dV.reshape(V.shape)

def run_optimized(Q, K, V, dO, M_data, N_data, band_width=256):
    B, H, M, D = Q.shape
    C = M_data.shape[-1]
    _Q = Q.reshape(B*H, M, D)
    _K = K.reshape(B*H, K.shape[2], D)
    _V = V.reshape(B*H, V.shape[2], V.shape[3])
    _dO = dO.reshape(B*H, dO.shape[2], dO.shape[3])
    _M_data = M_data.reshape(B*H, M, C)
    _N_data = N_data.reshape(B*H, N_data.shape[2], C)

    fn = lambda q, k, v, do, m_data, n_data: pallas_sinkhorn_bwd_fused_launcher(
        q, k, v, do, m_data, n_data, band_width, dq_kernel_multi_opt, dk_dv_kernel_multi_opt
    )
    dQ, dK, dV = jax.lax.map(lambda inputs: fn(*inputs), (_Q, _K, _V, _dO, _M_data, _N_data))
    
    return dQ.reshape(Q.shape), dK.reshape(K.shape), dV.reshape(V.shape)

jit_run_orig = jax.jit(run_original)
jit_run_opt = jax.jit(run_optimized)


In [20]:
# Generate Dummy Data
B, H, M, N, D, E = 2, 4, 16384, 16384, 64, 64
Q = jax.random.normal(jax.random.PRNGKey(0), (B, H, M, D))
K = jax.random.normal(jax.random.PRNGKey(1), (B, H, N, D))
V = jax.random.normal(jax.random.PRNGKey(2), (B, H, N, E))
dO = jax.random.normal(jax.random.PRNGKey(3), (B, H, M, E))
M_data = jax.random.normal(jax.random.PRNGKey(4), (B, H, M, 8)) * 0.1 * 0.1
N_data = jax.random.normal(jax.random.PRNGKey(5), (B, H, N, 8)) * 0.1 * 0.1
W = 512

# Correctness Check
dq_o, dk_o, dv_o = jit_run_orig(Q, K, V, dO, M_data, N_data, W)
dq_op, dk_op, dv_op = jit_run_opt(Q, K, V, dO, M_data, N_data, W)

print("Max diff dQ: ", jnp.max(jnp.abs(dq_o - dq_op)))
print("Max diff dK: ", jnp.max(jnp.abs(dk_o - dk_op)))
print("Max diff dV: ", jnp.max(jnp.abs(dv_o - dv_op)))

# Benchmark Original
jax.block_until_ready(jit_run_orig(Q, K, V, dO, M_data, N_data, W))
t0 = time.time()
for _ in range(1000):
    jax.block_until_ready(jit_run_orig(Q, K, V, dO, M_data, N_data, W))
print("Original 100 iters time: {:.4f}s".format(time.time() - t0))

# Benchmark Optimized
jax.block_until_ready(jit_run_opt(Q, K, V, dO, M_data, N_data, W))
t0 = time.time()
for _ in range(1000):
    jax.block_until_ready(jit_run_opt(Q, K, V, dO, M_data, N_data, W))
print("Optimized 100 iters time: {:.4f}s".format(time.time() - t0))


Max diff dQ:  0.0
Max diff dK:  0.0
Max diff dV:  0.0
Original 100 iters time: 1.2816s
Optimized 100 iters time: 1.2878s


In [21]:
import numpy as jnp

def _poly_exp2_frac(r):
    p5 = 1.5252715e-4
    p4 = 1.3215486e-3
    p3 = 9.6178085e-3
    p2 = 5.5504109e-2
    p1 = 2.4022651e-1
    p0 = 6.9314718e-1
    return 1.0 + r * (p0 + r * (p1 + r * (p2 + r * (p3 + r * (p4 + r * p5)))))

def safe_exp(x):
    LOG2E = jnp.float32(1.4426950408889634)
    t = x * LOG2E
    k = jnp.floor(t)
    r = t - k
    k = jnp.clip(k, -126.0, 127.0)
    pow2_int = jnp.exp2(k)
    pow2_frac = _poly_exp2_frac(r)
    return pow2_int * pow2_frac

# Simulate the dummy data values
jnp.random.seed(0)
s = jnp.random.normal(0, 1, (2, 4, 1024, 1024))
u1 = jnp.random.normal(0, 1, (2, 4, 1024, 1))
v1 = jnp.random.normal(0, 1, (2, 4, 1, 1024))
u2 = jnp.random.normal(0, 1, (2, 4, 1024, 1))

val1 = safe_exp(s + u2 + v1)
val2 = safe_exp(s + u1 + v1) * jnp.exp(u2 - u1)

max_abs_diff = jnp.max(jnp.abs(val1 - val2))
mean_abs_diff = jnp.mean(jnp.abs(val1 - val2))
max_val = jnp.max(val1)

print(f"Max Val: {max_val}")
print(f"Max Abs Diff: {max_abs_diff}")
print(f"Mean Abs Diff: {mean_abs_diff}")


Max Val: 4748.442457595602
Max Abs Diff: 0.03681406741088722
Mean Abs Diff: 1.563839323317062e-05
